# 20 — te_origin conditionné au montant (compte × tranche)

Idée : le taux de fraude d'un compte **diffère selon le montant**. Un compte qui ne fraude que
sur de gros montants a un profil que `te_origin` (1 scalaire) ne capture pas. On ajoute le
**target encoding de (compte × tranche de montant)** — une info que l'arbre ne peut pas dériver
seul (agrégation de labels conditionnelle). Fold-safe.

Réf 08 : recent2 **0.3662** | last 0.3607. Critère = recent2. Boussole : LB ≈ last − 0.004.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
from src import config as C
from src.validation import time_folds, evaluate_ap
from src.utils import op03_mask, seed_everything, make_submission
from src.features.temporal import balance_features, recency_features
from src.features.behavioral import behavioral_features
from src.encoding import oof_target_encode_train, fit_target_map, apply_target_map, recent_target_rate
seed_everything(42)
DATA = ROOT / "data"
train = pd.read_csv(DATA / "train.csv"); test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")
op03 = op03_mask(train).to_numpy(); y_all = train[C.TARGET].to_numpy()
folds_full = list(time_folds(train[C.PERIOD]))

# tranche de montant : médiane des montants op_03 (pas de label -> pas de fuite)
AMT_MED = float(np.median(train.loc[op03, C.AMOUNT]))
print("médiane montant op_03 :", round(AMT_MED, 2))
for d in (train, test):
    d["_amtb"] = (d[C.AMOUNT] >= AMT_MED).astype(int)
    d["_oxb"] = d[C.ORIGIN_ACCT].astype(str) + "_" + d["_amtb"].astype(str)

In [ ]:
EPS = 1e-6; WINDOWS = (5, 10, 20); SM = 30; SM_COND = 50
def row_features(df):
    f = pd.DataFrame(index=df.index)
    f["amount_log1p"] = np.log1p(np.maximum(df[C.AMOUNT], 0))
    f["amount_vs_origin_before"] = df[C.AMOUNT] / (np.abs(df[C.ORIGIN_BAL_BEFORE]) + EPS)
    f["amount_vs_dest_before"] = df[C.AMOUNT] / (np.abs(df[C.DEST_BAL_BEFORE]) + EPS)
    f["origin_balance_before"] = df[C.ORIGIN_BAL_BEFORE]; f["dest_balance_before"] = df[C.DEST_BAL_BEFORE]
    return pd.concat([f, balance_features(df)], axis=1)
def add_freq(X, src_df, ref_df):
    X = X.copy()
    for col in [C.ORIGIN_ACCT, C.DEST_ACCT]:
        freq = ref_df[col].value_counts(normalize=True)
        X[f"freq_{col}"] = src_df[col].map(freq).fillna(0).values
    return X
def base_build(df, ref):
    X = row_features(df).reset_index(drop=True)
    X = add_freq(X, df.reset_index(drop=True), ref)
    beh = behavioral_features(df, ref).reset_index(drop=True)
    rec = recency_features(df, ref).reset_index(drop=True)
    rt = recent_target_rate(df, ref, C.ORIGIN_ACCT, C.PERIOD, C.TARGET, WINDOWS).reset_index(drop=True)
    return pd.concat([X, beh, rec, rt], axis=1)
def feats_train(df, ref, cond):
    X = base_build(df, ref)
    X["te_origin"] = oof_target_encode_train(ref, C.ORIGIN_ACCT, C.TARGET, smoothing=SM)
    if cond:
        X["te_oxb"] = oof_target_encode_train(ref, "_oxb", C.TARGET, smoothing=SM_COND)
    return X
def feats_apply(df, ref, cond):
    X = base_build(df, ref)
    mp, gm = fit_target_map(ref, C.ORIGIN_ACCT, C.TARGET, smoothing=SM)
    X["te_origin"] = apply_target_map(df, C.ORIGIN_ACCT, mp, gm)
    if cond:
        mp2, gm2 = fit_target_map(ref, "_oxb", C.TARGET, smoothing=SM_COND)
        X["te_oxb"] = apply_target_map(df, "_oxb", mp2, gm2)
    return X
def make_cat():
    from catboost import CatBoostClassifier
    return CatBoostClassifier(loss_function="Logloss", eval_metric="PRAUC", depth=6,
                              learning_rate=0.05, iterations=600, random_seed=42, verbose=False)

## A/B : 08 (te_origin) vs + te_origin×montant

In [ ]:
def run_cv(cond):
    oof = np.zeros(len(train)); pf = []; lm = lc = None
    for tr_idx, va_idx in folds_full:
        tr_op = tr_idx[op03[tr_idx]]; va_op = va_idx[op03[va_idx]]; ref = train.iloc[tr_op]
        m = make_cat().fit(feats_train(train.iloc[tr_op], ref, cond), y_all[tr_op])
        oof[va_op] = m.predict_proba(feats_apply(train.iloc[va_op], ref, cond))[:, 1]
        pf.append(evaluate_ap(y_all[va_op], oof[va_op])); lm, lc = m, feats_apply(train.iloc[va_op], ref, cond).columns
    return pf, lm, lc

pf0, _, _ = run_cv(False)
pf1, m1, c1 = run_cv(True)
print(f"08 (te_origin)        recent2 {np.mean(pf0[-2:]):.4f} | last {pf0[-1]:.4f}")
print(f"+ te_origin×montant   recent2 {np.mean(pf1[-2:]):.4f} | last {pf1[-1]:.4f}" + ("  <-- BAT 08" if np.mean(pf1[-2:]) > np.mean(pf0[-2:]) else ""))
print(f"\nper-fold (+cond) : {[round(x,4) for x in pf1]}")
imp = m1.get_feature_importance() if hasattr(m1, 'get_feature_importance') else m1.feature_importances_
print("\nimportance te_oxb :", round(pd.Series(imp, index=c1)["te_oxb"], 2), "| te_origin :", round(pd.Series(imp, index=c1)["te_origin"], 2))

## Soumission (si recent2 > celui du 08)

In [ ]:
ref_full = train.iloc[np.where(op03)[0]]; yf = y_all[op03]
final = make_cat().fit(feats_train(ref_full, ref_full, True), yf)
te_op = op03_mask(test).to_numpy(); test_op = test.iloc[np.where(te_op)[0]]
proba = final.predict_proba(feats_apply(test_op, ref_full, True))[:, 1]
full = np.zeros(len(test)); full[te_op] = proba
path = make_submission(test[C.ID], full, "20_te_origin_amount")
sub = pd.read_csv(path)
assert list(sub.columns) == ["id", "target"] and len(sub) == len(test)
assert set(sub["id"]) == set(sample["id"]) and sub["target"].between(0, 1).all()
print("soumission écrite :", path, "| proba>0 :", int((sub['target'] > 0).sum()))